# RLSF — checkpoint selection and val inference


---
## 1 — Host, working tree, disk

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path

PY = sys.executable
ROOT = Path.cwd()
print('kernel', PY)

kernel /home/prnamhr/projects/Style-Aware-MT/.venv/bin/python


In [2]:
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv

name, memory.total [MiB], memory.used [MiB]
NVIDIA GeForce RTX 4090, 24564 MiB, 1 MiB


In [2]:
from pathlib import Path

if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone --branch feat/rlsf-implementation https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

/home/prnamhr/projects/Style-Aware-MT/notebooks/Style-Aware-MT
git@github.com: Permission denied (publickey).
fatal: Could not read from remote repository.

Please make sure you have the correct access rights
and the repository exists.
cbd4013


---
## 2 — Environment and run parameters

In [3]:
SKIP_KIWI = True

# The full dev slice, 499 segments. A cap here is a deviation from the documented selection path
# and belongs in the pre-registration if it is used.
DEV_LIMIT = 0

# Hours booked on this box, for the section 4 projection.
BUDGET_H = 8.0

KIWI_FLAG = '--skip_kiwi' if SKIP_KIWI else ''
print(f'skip_kiwi={SKIP_KIWI}  dev_limit={DEV_LIMIT}  budget={BUDGET_H} h')

skip_kiwi=True  dev_limit=0  budget=8.0 h


In [5]:
# %pip installs into the kernel; !pip may not.
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [17]:
COMET_PY = '.venv-comet/bin/python'

# Only needed if the selection table is to carry the Kiwi column. COMET's pins downgrade
# transformers and numpy, so it gets its own interpreter and never the kernel's.
if not SKIP_KIWI and not Path(COMET_PY).exists():
    pip = [COMET_PY, '-m', 'pip', 'install']
    subprocess.run([PY, '-m', 'venv', '.venv-comet'], check=True)
    subprocess.run([*pip, '--upgrade', 'pip'], check=True)
    subprocess.run([*pip, 'setuptools<81'], check=True)
    subprocess.run([*pip, '-r', 'requirements-comet.txt'], check=True)
if not SKIP_KIWI:
    subprocess.run([COMET_PY, '-c', 'import comet; print("comet ok")'], check=True)
else:
    print('no COMET worker this session')

no COMET worker this session


In [6]:
import getpass
import logging
import os

# HF_TOKEN only: the model repo is private and the base model is public. No rater key is read
# in this notebook, because nothing here can spend.
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY'):
    assert not os.environ.get(var), f'{var} is set; Phase B makes no paid call'
logging.getLogger('httpx').setLevel(logging.WARNING)
print('HF_TOKEN set, no rater keys present')

HF_TOKEN:  ········


HF_TOKEN set, no rater keys present


---
## 3 — Adapters

In [7]:
import json

from src.rlsf.config import load_config
from src.rlsf.select import checkpoints
from src.rlsf.train import arm_path, sidecar

CONFIG = 'configs/rlsf.yaml'
cfg = load_config(CONFIG, require_caps=False)

# The three arms, in the order the pre-registration reports them.
ARMS = {'RL-Metric': 'w3_0.0', 'RLSF-Judge': 'w3_2.0', 'RLSF-Judge-High': 'w3_6.0'}

FOUND = {}
for name, cell in ARMS.items():
    FOUND[cell] = checkpoints(arm_path(cfg['output']['adapter_dir'], cell))
    tags = [t for t, _ in FOUND[cell]]
    print(f"{name:16s} {cell:8s} {len(tags):3d} adapters: {', '.join(tags[:3])} ... {tags[-1]}")

counts = {c: len(v) for c, v in FOUND.items()}
# The arms are selected over the same ladder or the selections are not comparable.
assert len(set(counts.values())) == 1, counts
assert all(v[-1][0] == 'final' for v in FOUND.values()), 'an arm has no final adapter'

RL-Metric        w3_0.0    13 adapters: step100, step200, step300 ... final
RLSF-Judge       w3_2.0    13 adapters: step100, step200, step300 ... final
RLSF-Judge-High  w3_6.0    13 adapters: step100, step200, step300 ... final


In [8]:
# What came down must be what was trained: the manifests are committed, the weights are not.
for name, cell in ARMS.items():
    log = arm_path(cfg['output']['step_log'], cell)
    man = json.loads(sidecar(log, 'manifest.json').read_text())
    out, arm_dir = man['outcome'], arm_path(cfg['output']['adapter_dir'], cell)
    assert out['adapter_dir'] == str(arm_dir), out['adapter_dir']
    assert out['rollouts'] == 300, out['rollouts']
    print(f"{name:16s} {out['rollouts']} rollouts, omega {man['omega']}, "
          f"adapter {out['adapter_delta']['rel']:.3e} from init, halted={out['stop_reason']}")

RL-Metric        300 rollouts, omega {'bleu': 0.707107, 'kiwi': 0.707107, 'judge': 0.0}, adapter 9.305e-03 from init, halted=None
RLSF-Judge       300 rollouts, omega {'bleu': 0.408248, 'kiwi': 0.408248, 'judge': 0.816497}, adapter 9.337e-03 from init, halted=None
RLSF-Judge-High  300 rollouts, omega {'bleu': 0.162221, 'kiwi': 0.162221, 'judge': 0.973329}, adapter 9.396e-03 from init, halted=None


---
## 4 — The gate

In [9]:
N_DEV = sum(1 for line in open(cfg['data']['dev_file']) if line.strip())
N_VAL = sum(1 for line in open('data/splits/val.jsonl') if line.strip())

n_loads = sum(len(v) for v in FOUND.values()) + len(ARMS)
n_select = sum(len(v) for v in FOUND.values()) * (DEV_LIMIT or N_DEV)
n_val = len(ARMS) * N_VAL

print(f"selection  {sum(len(v) for v in FOUND.values())} checkpoints x {DEV_LIMIT or N_DEV} dev "
      f"segments = {n_select:,} generations")
print(f"inference  {len(ARMS)} arms x {N_VAL} val segments = {n_val:,} generations")
print(f"total      {n_select + n_val:,} greedy generations, {n_loads} model loads")

selection  39 checkpoints x 499 dev segments = 19,461 generations
inference  3 arms x 1323 val segments = 3,969 generations
total      23,430 greedy generations, 42 model loads


In [10]:
import time

from src.infer.run import build_zeroshot_user, make_client

# Timed outside outputs/rlsf/select_*/ on purpose: a short file written there would be skipped
# by the real pass (select.py only regenerates under --overwrite) and freeze a truncated
# checkpoint into the selection.
PROBE_N = 8
style = Path(cfg['prompt']['style_instruction_file']).read_text(encoding='utf-8')
dev = [json.loads(x) for x in open(cfg['data']['dev_file']) if x.strip()][:PROBE_N]
val = [json.loads(x) for x in open('data/splits/val.jsonl') if x.strip()][:PROBE_N]

# max_tokens 1024 is the val setting; selection runs at the arm's 192, so the dev rate this
# yields is an over-estimate and the projection errs long.
probe_gen = {**cfg['generator'], 'temperature': 0.0, 'top_p': 1.0, 'max_tokens': 1024,
             'seed': cfg['rlsf']['seed'], 'adapter_path': str(dict(FOUND['w3_0.0'])['final'])}

t0 = time.perf_counter()
probe = make_client(probe_gen)
load_s = time.perf_counter() - t0


def rate(client, rows):
    t = time.perf_counter()
    for row in rows:
        client.complete(style, build_zeroshot_user(row['input']))
    return (time.perf_counter() - t) / len(rows)


dev_s, val_s = rate(probe, dev), rate(probe, val)
print(f'{load_s:.0f}s model load')
print(f'{dev_s:.2f}s per dev segment, {val_s:.2f}s per val segment')

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

46s model load
1.83s per dev segment, 2.24s per val segment


In [11]:
select_h = (n_select * dev_s + (n_loads - len(ARMS)) * load_s) / 3600
infer_h = (n_val * val_s + len(ARMS) * load_s) / 3600
print(f'selection  {select_h:5.1f} h')
print(f'inference  {infer_h:5.1f} h')
print(f'total      {select_h + infer_h:5.1f} h against {BUDGET_H:.1f} h booked')

if select_h + infer_h > 0.8 * BUDGET_H:
    print('\nThis does not fit with room to spare. Pull a lever below before section 5.')
else:
    print('\nFits. Section 5 may start.')

selection   10.4 h
inference    2.5 h
total       12.9 h against 8.0 h booked

This does not fit with room to spare. Pull a lever below before section 5.


---
## 5 — Selection, one arm per cell

In [27]:
!{PY} manage.py rlsf_select --config {CONFIG} --cell w3_0.0 --dev-limit {DEV_LIMIT} {KIWI_FLAG}

13 checkpoints in models/rlsf_grpo_w3_0.0, 499 dev segments each
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 138.15it/s]
[step100] generating 499 dev translations with models/rlsf_grpo_w3_0.0/checkpoint-100 ...
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 134.61it/s]
[step200] generating 499 dev translations with models/rlsf_grpo_w3_0.0/checkpoint-200 ...
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 131.52it/s]
[step300] generating 499 dev translations with models/rlsf_grpo_w3_0.0/checkpoint-300 ...
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 132.65it/s]
[step400] generating 499 dev translations with models/rlsf_grpo_w3_0.0/checkpoint-400 ...
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 145.70it/s]
[step500] generating 499 dev translations with models/rlsf_grpo_w3_0.0/checkpoint-500 ...
Loading weights: 100%|██████████████████████| 339/339 [00:03<00:00, 108.49it/s]
[step

In [14]:
!{PY} manage.py rlsf_select --config {CONFIG} --cell w3_2.0 --dev-limit {DEV_LIMIT} {KIWI_FLAG}

13 checkpoints in models/rlsf_grpo_w3_2.0, 499 dev segments each
skip step100: outputs/rlsf/select_w3_2.0/step100_dev.jsonl exists (--overwrite to regenerate)
skip step200: outputs/rlsf/select_w3_2.0/step200_dev.jsonl exists (--overwrite to regenerate)
skip step300: outputs/rlsf/select_w3_2.0/step300_dev.jsonl exists (--overwrite to regenerate)
skip step400: outputs/rlsf/select_w3_2.0/step400_dev.jsonl exists (--overwrite to regenerate)
Loading weights: 100%|██████████████████████| 339/339 [00:03<00:00, 112.60it/s]
[step500] generating 499 dev translations with models/rlsf_grpo_w3_2.0/checkpoint-500 ...
Loading weights: 100%|██████████████████████| 339/339 [00:03<00:00, 112.12it/s]
[step600] generating 499 dev translations with models/rlsf_grpo_w3_2.0/checkpoint-600 ...
Loading weights: 100%|██████████████████████| 339/339 [00:03<00:00, 112.62it/s]
[step700] generating 499 dev translations with models/rlsf_grpo_w3_2.0/checkpoint-700 ...
Loading weights: 100%|██████████████████████| 339

In [15]:
!{PY} manage.py rlsf_select --config {CONFIG} --cell w3_6.0 --dev-limit {DEV_LIMIT} {KIWI_FLAG}

13 checkpoints in models/rlsf_grpo_w3_6.0, 499 dev segments each
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 139.44it/s]
[step100] generating 499 dev translations with models/rlsf_grpo_w3_6.0/checkpoint-100 ...
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 137.08it/s]
[step200] generating 499 dev translations with models/rlsf_grpo_w3_6.0/checkpoint-200 ...
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 137.63it/s]
[step300] generating 499 dev translations with models/rlsf_grpo_w3_6.0/checkpoint-300 ...
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 135.17it/s]
[step400] generating 499 dev translations with models/rlsf_grpo_w3_6.0/checkpoint-400 ...
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 137.06it/s]
[step500] generating 499 dev translations with models/rlsf_grpo_w3_6.0/checkpoint-500 ...
Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 134.26it/s]
[step

---
## 6 — Freeze the selection into the eval configs

Done here rather than by hand. A hand-edited `adapter_path` under time pressure on a rented box
is how an arm ends up scored on another arm's checkpoint, and the val hypotheses carry no record
of which adapter wrote them beyond this line.

In [16]:
import re

SELECTED = {}
for cell in ARMS.values():
    sel = json.loads(Path(f'results/rlsf_select_{cell}.json').read_text())
    path = Path(sel['selected_path'])
    arm_dir = arm_path(cfg['output']['adapter_dir'], cell)
    assert (path / 'adapter_config.json').exists(), path
    # The selected checkpoint must belong to this arm, not the one selected before it.
    assert path == arm_dir or arm_dir in path.parents, (cell, path)

    cfg_path = Path(f'configs/rlsf_eval_{cell}.yaml')
    text = cfg_path.read_text(encoding='utf-8')
    new, n = re.subn(
        r'^(\s*adapter_path:\s*)\S+.*$',
        rf'\g<1>{path}    # {sel["selected"]}, selected on the dev slice',
        text, count=1, flags=re.M,
    )
    assert n == 1, f'no adapter_path line in {cfg_path}'
    cfg_path.write_text(new, encoding='utf-8')
    SELECTED[cell] = sel
    print(f"{cell:8s} {sel['selected']:8s} chrF {sel['rows'][0]['chrF']:6.2f}  ->  {cfg_path}")

w3_0.0   step200  chrF  63.67  ->  configs/rlsf_eval_w3_0.0.yaml
w3_2.0   step200  chrF  63.57  ->  configs/rlsf_eval_w3_2.0.yaml
w3_6.0   step100  chrF  63.61  ->  configs/rlsf_eval_w3_6.0.yaml


In [17]:
# The rule's own ordering, per arm: the chrF band first, then held-out register distance.
for cell, sel in SELECTED.items():
    by_tag = {r['tag']: r for r in sel['rows']}
    print(f"\n{cell}  ({sel['n_segments']} dev segments, margin {sel['adequacy_margin']})")
    print(f"  {'tag':8s} {'chrF':>7s} {'BLEU':>7s} {'dist_heldout':>13s} {'dist_reward':>12s}")
    for tag in sel['ranked'][:5]:
        r = by_tag[tag]
        mark = ' <-' if tag == sel['selected'] else ''
        print(f"  {tag:8s} {r['chrF']:7.2f} {r['BLEU']:7.2f} {r['dist_heldout']:13.4f} "
              f"{r['dist_reward']:12.4f}{mark}")


w3_0.0  (499 dev segments, margin 1.0)
  tag         chrF    BLEU  dist_heldout  dist_reward
  step200    63.84   48.79        0.5907       0.1907 <-
  step100    63.67   48.58        0.5910       0.1916
  step900    63.33   48.16        0.6032       0.1997
  step400    63.88   49.05        0.6035       0.1884
  step300    63.76   48.95        0.6045       0.1906

w3_2.0  (499 dev segments, margin 1.0)
  tag         chrF    BLEU  dist_heldout  dist_reward
  step200    63.91   48.81        0.6083       0.1837 <-
  step100    63.57   48.23        0.6144       0.1900
  step300    63.78   48.56        0.6237       0.1887
  step400    63.95   48.82        0.6319       0.1903
  step500    63.52   48.29        0.6524       0.1976

w3_6.0  (499 dev segments, margin 1.0)
  tag         chrF    BLEU  dist_heldout  dist_reward
  step100    63.61   48.49        0.5965       0.1866 <-
  step200    63.82   48.68        0.6213       0.1869
  step300    63.82   48.59        0.6347       0.1900
  step4

---
## 7 — Val inference

In [20]:
!{PY} manage.py infer --condition peft --config configs/rlsf_eval_w3_0.0.yaml

Loading weights: 100%|██████████████████████| 339/339 [00:01<00:00, 171.94it/s]
Output name overridden: condition 'peft' -> outputs/rlsf_w3_0.0_val.jsonl
Generating 1323 translations with Qwen/Qwen2.5-7B-Instruct (peft) ...
  5/1323
  10/1323
  15/1323
  20/1323
  25/1323
  30/1323
  35/1323
  40/1323
  45/1323
  50/1323
  55/1323
  60/1323
  65/1323
  70/1323
  75/1323
  80/1323
  85/1323
  90/1323
  95/1323
  100/1323
  105/1323
  110/1323
  115/1323
  120/1323
  125/1323
  130/1323
  135/1323
  140/1323
  145/1323
  150/1323
  155/1323
  160/1323
  165/1323
  170/1323
  175/1323
  180/1323
  185/1323
  190/1323
  195/1323
  200/1323
  205/1323
  210/1323
  215/1323
  220/1323
  225/1323
  230/1323
  235/1323
  240/1323
  245/1323
  250/1323
  255/1323
  260/1323
  265/1323
  270/1323
  275/1323
  280/1323
  285/1323
  290/1323
  295/1323
  300/1323
  305/1323
  310/1323
  315/1323
  320/1323
  325/1323
  330/1323
  335/1323
  340/1323
  345/1323
  350/1323
  355/1323
  360/1323
  36

In [21]:
!{PY} manage.py infer --condition peft --config configs/rlsf_eval_w3_2.0.yaml

Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 138.96it/s]
Output name overridden: condition 'peft' -> outputs/rlsf_w3_2.0_val.jsonl
Generating 1323 translations with Qwen/Qwen2.5-7B-Instruct (peft) ...
  5/1323
  10/1323
  15/1323
  20/1323
  25/1323
  30/1323
  35/1323
  40/1323
  45/1323
  50/1323
  55/1323
  60/1323
  65/1323
  70/1323
  75/1323
  80/1323
  85/1323
  90/1323
  95/1323
  100/1323
  105/1323
  110/1323
  115/1323
  120/1323
  125/1323
  130/1323
  135/1323
  140/1323
  145/1323
  150/1323
  155/1323
  160/1323
  165/1323
  170/1323
  175/1323
  180/1323
  185/1323
  190/1323
  195/1323
  200/1323
  205/1323
  210/1323
  215/1323
  220/1323
  225/1323
  230/1323
  235/1323
  240/1323
  245/1323
  250/1323
  255/1323
  260/1323
  265/1323
  270/1323
  275/1323
  280/1323
  285/1323
  290/1323
  295/1323
  300/1323
  305/1323
  310/1323
  315/1323
  320/1323
  325/1323
  330/1323
  335/1323
  340/1323
  345/1323
  350/1323
  355/1323
  360/1323
  36

In [22]:
!{PY} manage.py infer --condition peft --config configs/rlsf_eval_w3_6.0.yaml

Loading weights: 100%|██████████████████████| 339/339 [00:02<00:00, 139.12it/s]
Output name overridden: condition 'peft' -> outputs/rlsf_w3_6.0_val.jsonl
Generating 1323 translations with Qwen/Qwen2.5-7B-Instruct (peft) ...
  5/1323
  10/1323
  15/1323
  20/1323
  25/1323
  30/1323
  35/1323
  40/1323
  45/1323
  50/1323
  55/1323
  60/1323
  65/1323
  70/1323
  75/1323
  80/1323
  85/1323
  90/1323
  95/1323
  100/1323
  105/1323
  110/1323
  115/1323
  120/1323
  125/1323
  130/1323
  135/1323
  140/1323
  145/1323
  150/1323
  155/1323
  160/1323
  165/1323
  170/1323
  175/1323
  180/1323
  185/1323
  190/1323
  195/1323
  200/1323
  205/1323
  210/1323
  215/1323
  220/1323
  225/1323
  230/1323
  235/1323
  240/1323
  245/1323
  250/1323
  255/1323
  260/1323
  265/1323
  270/1323
  275/1323
  280/1323
  285/1323
  290/1323
  295/1323
  300/1323
  305/1323
  310/1323
  315/1323
  320/1323
  325/1323
  330/1323
  335/1323
  340/1323
  345/1323
  350/1323
  355/1323
  360/1323
  36

---
## 8 — Verify before teardown

In [23]:
import yaml

VAL = [json.loads(x) for x in open('data/splits/val.jsonl') if x.strip()]
HYPS = {}
for cell in ARMS.values():
    name = yaml.safe_load(Path(f'configs/rlsf_eval_{cell}.yaml').read_text())['output']['name']
    path = Path('outputs') / f'{name}_val.jsonl'
    rows = [json.loads(x) for x in open(path) if x.strip()]
    HYPS[cell] = (path, rows)

    assert len(rows) == len(VAL), f'{path}: {len(rows)} rows, expected {len(VAL)}'
    assert all(a['input'] == b['input'] for a, b in zip(rows, VAL)), f'{path}: source misalignment'
    errors = [r for r in rows if r.get('error')]
    empty = [r for r in rows if not r['prediction'].strip()]
    assert not errors, f'{path}: {len(errors)} segments recorded an error'
    assert not empty, f'{path}: {len(empty)} empty predictions'
    assert {r['condition'] for r in rows} == {'peft'}
    print(f'{path}  {len(rows)} rows, no errors, no empties')

outputs/rlsf_w3_0.0_val.jsonl  1323 rows, no errors, no empties
outputs/rlsf_w3_2.0_val.jsonl  1323 rows, no errors, no empties
outputs/rlsf_w3_6.0_val.jsonl  1323 rows, no errors, no empties


In [24]:
from sacrebleu.metrics import CHRF

from src.eval.quick import _marker_rate

chrf = CHRF()
for cell, (path, rows) in HYPS.items():
    preds = [r['prediction'] for r in rows]
    refs = [r['output'] for r in rows]
    print(f"{cell:8s} chrF {chrf.corpus_score(preds, [refs]).score:6.2f}  "
          f"marker_rate {_marker_rate(preds):5.2f}  "
          f"adapter {SELECTED[cell]['selected_path']}")

w3_0.0   chrF  41.85  marker_rate  0.93  adapter models/rlsf_grpo_w3_0.0/checkpoint-200
w3_2.0   chrF  42.04  marker_rate  0.99  adapter models/rlsf_grpo_w3_2.0/checkpoint-200
w3_6.0   chrF  42.08  marker_rate  0.95  adapter models/rlsf_grpo_w3_6.0/checkpoint-100


In [25]:
# The seal: nothing in this session may have read the test split.
for cell, (path, _) in HYPS.items():
    assert 'test' not in path.name, path
    u = json.loads(path.with_name(f'{path.stem}_usage.json').read_text())
    assert u['cost_usd'] == 0.0, u
    print(f"{cell:8s} {u['calls']} local calls, ${u['cost_usd']:.2f}")
print('\n0 paid calls, $0.00 spent in Phase B')

w3_0.0   1323 local calls, $0.00
w3_2.0   1323 local calls, $0.00
w3_6.0   1323 local calls, $0.00

0 paid calls, $0.00 spent in Phase B
